### Reglungsnormalform und Steuerbarkeit


In [2]:
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
from IPython.display import display, Math

def interact_dgl_rnf(a0=2.0, a1=3.0, a2=1.0, b=1.0):
    r"""
    SISO 3. Ordnung:
    y''' + a2*y'' + a1*y' + a0*y = b*u
    """
    # 1. Matrizen der RNF aufstellen
    A_R = np.array([
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
        [-a0, -a1, -a2]
    ])
    b_R = np.array([[0.0], [0.0], [b]])

    # 2. Steuerbarkeitsmatrix S_s = [b_R, A_R*b_R, A_R^2*b_R] berechnen
    Ab_R = A_R @ b_R
    A2b_R = A_R @ Ab_R
    S_s = np.hstack([b_R, Ab_R, A2b_R])

    # Status der Steuerbarkeit prüfen
    if abs(np.linalg.det(S_s)) > 1e-6:
        status_str = r"\text{(steuerbar, Rang = 3)}"
    else:
        status_str = r"\text{\color{red}{(nicht steuerbar, Rang < 3)}}"

    # 3. Formatierung der DGL-Gleichung
    dgl_left = r"y'''"
    if a2 != 0: dgl_left += f" + {a2:.2f}y''" if a2 > 0 else f" - {abs(a2):.2f}y''"
    if a1 != 0: dgl_left += f" + {a1:.2f}y'" if a1 > 0 else f" - {abs(a1):.2f}y'"
    if a0 != 0: dgl_left += f" + {a0:.2f}y" if a0 > 0 else f" - {abs(a0):.2f}y"

    dgl_right = f"{b:.2f}u(t)"

    # 4. Zweispaltiges, beidseitig linksbündiges LaTeX-Raster (l@{\quad}l)
    latex_out = r"""
    \begin{array}{l@{\quad}l}
    \text{\textbf{1. Differentialgleichung:}} & %s = %s \\[12pt]
    \text{\textbf{2. Regelungsnormalform:}} &
    \begin{bmatrix} \dot{x}_1 \\ \dot{x}_2 \\ \dot{x}_3 \end{bmatrix} =
    \underbrace{\begin{bmatrix} 0 & 1 & 0 \\ 0 & 0 & 1 \\ \mathbf{%.2f} & \mathbf{%.2f} & \mathbf{%.2f} \end{bmatrix}}_{\mathbf{A}_R}
    \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} +
    \underbrace{\begin{bmatrix} 0 \\ 0 \\ \mathbf{%.2f} \end{bmatrix}}_{\mathbf{b}_R} u(t), \quad
    y(t) = \underbrace{\begin{bmatrix} 1 & 0 & 0 \end{bmatrix}}_{\mathbf{c}_R^T} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} \\[12pt]
    \text{\textbf{3. Steuerbarkeitsmatrix }} \mathbf{S}_s\text{\textbf{:}} &
    \mathbf{S}_s = \begin{bmatrix} \mathbf{b}_R & \mathbf{A}_R\mathbf{b}_R & \mathbf{A}_R^2\mathbf{b}_R \end{bmatrix} =
    \begin{bmatrix}
    %.2f & %.2f & %.2f \\
    %.2f & %.2f & %.2f \\
    %.2f & %.2f & %.2f
    \end{bmatrix} \quad %s
    \end{array}
    """ % (
        dgl_left, dgl_right,
        -a0, -a1, -a2, b,
        S_s[0,0], S_s[0,1], S_s[0,2],
        S_s[1,0], S_s[1,1], S_s[1,2],
        S_s[2,0], S_s[2,1], S_s[2,2], status_str
    )

    display(Math(latex_out))

# Layout-Anpassung für die Slider
style = {'description_width': 'initial'}
slider_layout = widgets.Layout(width='420px')

# Interaktive Schieberegler
interact(
    interact_dgl_rnf,
    a0=FloatSlider(min=-5.0, max=10.0, step=0.5, value=2.0, description=r'a0 (y)', style=style, layout=slider_layout),
    a1=FloatSlider(min=-5.0, max=10.0, step=0.5, value=3.0, description=r'a1 (y\')', style=style, layout=slider_layout),
    a2=FloatSlider(min=-5.0, max=10.0, step=0.5, value=1.0, description=r'a2 (y\'\')', style=style, layout=slider_layout),
    b=FloatSlider(min=-5.0, max=10.0, step=0.5, value=1.0, description=r'b (u)', style=style, layout=slider_layout)
);

interactive(children=(FloatSlider(value=2.0, description='a0 (y)', layout=Layout(width='420px'), max=10.0, min…

### Reglerentwurf im Zustandsraum


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
from IPython.display import display, Math

def plot_pole_placement(re_pole=-2.0, im_pole=3.0):
    # Ursprüngliches instabiles/schwach gedämpftes System (2. Ordnung)
    A = np.array([[0.0, 1.0], [2.0, 0.5]])
    b = np.array([[0.0], [1.0]])
    c = np.array([[1.0, 0.0]])
    d = np.array([[0.0]])

    # 1. Gewünschtes charakteristisches Polynom
    p1 = -2.0 * re_pole
    p0 = re_pole**2 + im_pole**2

    # 2. Matrixpolynom Pd(A)
    A2 = A @ A
    Pd_A = A2 + p1 * A + p0 * np.eye(2)

    # 3. Steuerbarkeitsmatrix S_s
    S_s = np.hstack([b, A @ b])
    S_s_inv = np.linalg.inv(S_s)

    # 4. Ackermann-Formel: K = e_n^T * S_s^-1 * Pd(A)
    e_n = np.array([0.0, 1.0])
    q_T = e_n @ S_s_inv
    K = (q_T @ Pd_A).reshape(1, 2)

    # Zweispaltiges, beidseitig linksbündiges LaTeX-Raster (l@{\quad}l)
    latex_out = r"""
    \begin{array}{l@{\quad}l}
    \text{\textbf{1. Wunschpolynom:}} & P_d(s) = s^2 + \mathbf{%.2f}\,s + \mathbf{%.2f} \\[10pt]
    \text{\textbf{2. Steuerbarkeitsmatrix:}} & \mathbf{S}_s = \begin{bmatrix} \mathbf{b} & \mathbf{A}\mathbf{b} \end{bmatrix} = \begin{bmatrix} %.2f & %.2f \\ %.2f & %.2f \end{bmatrix} \\[10pt]
    \text{\textbf{3. Ackermann-Formel:}} & \mathbf{K} = \mathbf{e}_2^T \mathbf{S}_s^{-1} P_d(\mathbf{A}) \\[6pt]
    & = \begin{bmatrix} 0 & 1 \end{bmatrix} \begin{bmatrix} %.2f & %.2f \\ %.2f & %.2f \end{bmatrix}^{-1} \left( \mathbf{A}^2 + %.2f \mathbf{A} + %.2f \mathbf{I} \right) \\[6pt]
    & \implies \mathbf{K} = \begin{bmatrix} \mathbf{%.2f} & \mathbf{%.2f} \end{bmatrix}
    \end{array}
    """ % (
        p1, p0,
        S_s[0,0], S_s[0,1], S_s[1,0], S_s[1,1],
        S_s[0,0], S_s[0,1], S_s[1,0], S_s[1,1],
        p1, p0,
        K[0,0], K[0,1]
    )
    display(Math(latex_out))

    # Geschlossenes System A_cl = A - b*K
    A_cl = A - b @ K
    V = -1.0 / (c @ np.linalg.inv(A_cl) @ b)[0, 0]

    sys_cl = signal.StateSpace(A_cl, b * V, c, d)
    t = np.linspace(0, 5, 500)
    t, y = signal.step(sys_cl, T=t)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    # Polort-Plot
    open_poles = np.linalg.eigvals(A)
    desired_poles = [complex(re_pole, im_pole), complex(re_pole, -im_pole)]

    ax1.plot(np.real(open_poles), np.imag(open_poles), 'rx', markersize=10, markeredgewidth=2, label='Pole offene Strecke')
    ax1.plot(np.real(desired_poles), np.imag(desired_poles), 'go', markersize=10, fillstyle='none', markeredgewidth=2, label='Wunschpole (geschlossen)')
    ax1.axvline(0, color='black', linestyle='--', alpha=0.7)
    ax1.axhline(0, color='black', linestyle='--', alpha=0.7)
    ax1.set_xlim([-10, 5])
    ax1.set_ylim([-8, 8])
    ax1.set_xlabel(r'Realteil $\mathrm{Re}(s)$')
    ax1.set_ylabel(r'Imaginärteil $\mathrm{Im}(s)$')
    ax1.set_title('Polort in s-Ebene')
    ax1.grid(True)
    ax1.legend()

    # Sprungantwort-Plot
    ax2.plot(t, y, 'b-', linewidth=2, label=r'Regelgröße $y(t)$')
    ax2.axhline(1.0, color='r', linestyle='--', label=r'Sollwert $w=1$')
    ax2.set_xlabel(r'Zeit $t$ [s]')
    ax2.set_ylabel(r'Ausgang $y(t)$')
    ax2.set_title('Sprungantwort des geschlossenen Kreises')
    ax2.grid(True)
    ax2.legend()

    plt.tight_layout()
    plt.show()

# Layout-Anpassung für die Slider
style = {'description_width': 'initial'}
slider_layout = widgets.Layout(width='420px')

interact(
    plot_pole_placement,
    re_pole=FloatSlider(min=-8.0, max=1.0, step=0.5, value=-2.0, description='Realteil', style=style, layout=slider_layout),
    im_pole=FloatSlider(min=0.0, max=8.0, step=0.5, value=3.0, description='Imaginärteil', style=style, layout=slider_layout)
);

interactive(children=(FloatSlider(value=-2.0, description='Realteil', layout=Layout(width='420px'), max=1.0, m…

### Vorfilter

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
from IPython.display import display, Math

def plot_prefilter_impact(manual_V_factor=1.0):
    # Streckendaten (System 2. Ordnung)
    A = np.array([[0.0, 1.0], [-2.0, -3.0]])
    b = np.array([[0.0], [1.0]])
    c = np.array([[1.0, 0.0]])

    # Gegebener Regler K für Wunschpole [-2, -4]
    placed = signal.place_poles(A, b, [-2.0, -4.0])
    K = placed.gain_matrix
    A_cl = A - b @ K

    # Statische Verstärkung des geschlossenen Kreises G_cl(0)
    dc_gain_cl = -(c @ np.linalg.inv(A_cl) @ b)[0, 0]

    # Exakter theoretischer Vorfilter V_exact
    V_exact = 1.0 / dc_gain_cl

    # Tatsächlich genutzter Vorfilter
    V_used = V_exact * manual_V_factor

    # Formel-Ausgabe gemäß Bild mit dem gewählten Wert
    latex_out = r"V = -\left( \mathbf{c}^T \cdot (\mathbf{A} - \mathbf{b}\mathbf{k}^T)^{-1} \cdot \mathbf{b} \right)^{-1} \cdot %.1f = \mathbf{%.2f}" % (manual_V_factor, V_used)
    display(Math(latex_out))

    # Simulation der Sprungantwort
    sys_cl = signal.StateSpace(A_cl, b * V_used, c, 0)
    t = np.linspace(0, 6, 500)
    t, y = signal.step(sys_cl, T=t)

    plt.figure(figsize=(9, 4.0))
    plt.plot(t, y, 'b-', linewidth=2, label=f'Ausgang')
    plt.axhline(1.0, color='r', linestyle='--', label='Führungsgröße')

    # Bleibenden Endwert einzeichnen
    y_inf = y[-1]
    plt.axhline(y_inf, color='gray', linestyle=':', label=f'Endwert')

    plt.title('Einfluss des Vorfilters auf die stationäre Genauigkeit')
    plt.xlabel('Zeit $t$ [s]')
    plt.ylabel('Signalhöhe')
    plt.ylim([0, 1.8])
    plt.grid(True)
    plt.legend()
    plt.show()

# Layout-Anpassung für den Slider
style = {'description_width': 'initial'}
slider_layout = widgets.Layout(width='450px')

interact(
    plot_prefilter_impact,
    manual_V_factor=FloatSlider(
        min=0.2, max=1.8, step=0.1, value=1.0,
        description='Vorfilter-Skalierung',
        style=style, layout=slider_layout
    )
);

interactive(children=(FloatSlider(value=1.0, description='Vorfilter-Skalierung', layout=Layout(width='450px'),…